# 中国进出口数据合并与整理

本notebook用于处理和合并中国进出口数据文件：

1. 读取数据：从`data/raw`目录读取所有Import和Export文件
2. 合并数据：分别合并所有Import和Export数据
3. 保存结果：将合并后的文件保存到`data/interim`目录

## 数据结构
每个文件包含以下列：
- date of data：数据日期
- commodity code：商品代码
- commodity：商品名称
- trading partner code：贸易伙伴代码
- trading partner：贸易伙伴名称
- quantity：数量
- unit：单位
- us dollar：美元价值

In [2]:
# 导入必要的库
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置数据路径
RAW_DATA_PATH = '../data/raw'
INTERIM_DATA_PATH = '../data/interim'

# 创建interim目录（如果不存在）
Path(INTERIM_DATA_PATH).mkdir(parents=True, exist_ok=True)

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## 数据读取与合并

读取raw目录下的所有Import和Export数据文件，并分别合并成两个数据框。

In [4]:
# 定义数据加载函数
def load_csv_with_encoding(file_path):
    """
    尝试使用不同的编码读取CSV文件
    """
    encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030']
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            return df
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"无法使用已知编码读取文件：{file_path}")

# 读取所有Import文件
print("=== 开始读取Import文件 ===")
import_files = glob.glob(os.path.join(RAW_DATA_PATH, '*Import.csv'))
import_dfs = []
for f in import_files:
    try:
        df = load_csv_with_encoding(f)
        df['source_file'] = os.path.basename(f)  # 添加来源文件信息
        import_dfs.append(df)
        print(f"成功读取：{os.path.basename(f)}")
    except Exception as e:
        print(f"读取文件失败 {os.path.basename(f)}: {str(e)}")
import_data = pd.concat(import_dfs, ignore_index=True)

# 读取所有Export文件
print("\n=== 开始读取Export文件 ===")
export_files = glob.glob(os.path.join(RAW_DATA_PATH, '*Export.csv'))
export_dfs = []
for f in export_files:
    try:
        df = load_csv_with_encoding(f)
        df['source_file'] = os.path.basename(f)  # 添加来源文件信息
        export_dfs.append(df)
        print(f"成功读取：{os.path.basename(f)}")
    except Exception as e:
        print(f"读取文件失败 {os.path.basename(f)}: {str(e)}")
export_data = pd.concat(export_dfs, ignore_index=True)

# 显示数据信息
print("\n=== 数据读取完成 ===")
print(f"Import数据形状：{import_data.shape}")
print(f"Export数据形状：{export_data.shape}")

# 显示数据列名
print("\nImport数据列名：")
print(import_data.columns.tolist())
print("\nExport数据列名：")
print(export_data.columns.tolist())

=== 开始读取Import文件 ===
成功读取：2024_China Import.csv
成功读取：202508_China Import.csv
成功读取：202501&06_China Import.csv
成功读取：202507_China Import.csv
成功读取：2020_China Import.csv
成功读取：202509_China Import.csv
成功读取：2021_China Import.csv
成功读取：2022_China Import.csv
成功读取：2023_China Import.csv

=== 开始读取Export文件 ===
成功读取：2021_China Export.csv
成功读取：202509_China Export.csv
成功读取：2020_China Export.csv
成功读取：202501&06_China Export.csv
成功读取：202507_China Export.csv
成功读取：2024_China Export.csv
成功读取：202508_China Export.csv
成功读取：2023_China Export.csv
成功读取：2022_China Export.csv

=== 数据读取完成 ===
Import数据形状：(1776, 12)
Export数据形状：(1164, 12)

Import数据列名：
['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'US dollar', 'Unnamed: 10', 'source_file']

Export数据列名：
['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'U

In [5]:
# 处理数据预处理函数
def preprocess_data(df, data_type='import'):
    # 创建数据副本
    df = df.copy()
    
    # 统一列名格式（去除空格，转换为小写）
    df.columns = [col.strip().lower() for col in df.columns]
    
    # 检查缺失值
    missing_values = df.isnull().sum()
    print(f"\n{data_type.capitalize()}数据缺失值统计：")
    print(missing_values[missing_values > 0])
    
    # 对数值列进行缺失值处理（用0填充）
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    df[numeric_columns] = df[numeric_columns].fillna(0)
    
    # 对分类列进行缺失值处理（用'未知'填充）
    categorical_columns = df.select_dtypes(include=['object']).columns
    df[categorical_columns] = df[categorical_columns].fillna('未知')
    
    return df

# 处理Import数据
processed_import_data = preprocess_data(import_data, 'import')

# 处理Export数据
processed_export_data = preprocess_data(export_data, 'export')

# 显示处理后的数据信息
print("\nImport数据信息：")
print(processed_import_data.info())
print("\nExport数据信息：")
print(processed_export_data.info())


Import数据缺失值统计：
unnamed: 10    1776
dtype: int64

Export数据缺失值统计：
unnamed: 10    1164
dtype: int64

Import数据信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1776 entries, 0 to 1775
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date of data            1776 non-null   int64  
 1   commodity code          1776 non-null   int64  
 2   commodity               1776 non-null   object 
 3   trading partner code    1776 non-null   int64  
 4   trading partner         1776 non-null   object 
 5   quantity                1776 non-null   int64  
 6   unit                    1776 non-null   object 
 7   supplementary quantity  1776 non-null   int64  
 8   supplementary unit      1776 non-null   object 
 9   us dollar               1776 non-null   object 
 10  unnamed: 10             1776 non-null   float64
 11  source_file             1776 non-null   object 
dtypes: float64(1), int64(5), object(6)


## 保存合并后的数据

将合并后的数据保存到interim目录。

## 3. 数据保存与统计

In [7]:
# 保存合并后的数据
import_output_path = os.path.join(INTERIM_DATA_PATH, 'combined_import_data.csv')
export_output_path = os.path.join(INTERIM_DATA_PATH, 'combined_export_data.csv')

# 移除可能存在的Unnamed列
import_cols = [col for col in import_data.columns if not col.startswith('Unnamed')]
export_cols = [col for col in export_data.columns if not col.startswith('Unnamed')]

import_data = import_data[import_cols]
export_data = export_data[export_cols]

# 保存数据
import_data.to_csv(import_output_path, index=False, encoding='utf-8-sig')
export_data.to_csv(export_output_path, index=False, encoding='utf-8-sig')

# 显示保存信息
print("=== 数据保存完成 ===")
print(f"Import数据已保存到：{import_output_path}")
print(f"Export数据已保存到：{export_output_path}")

# 显示文件大小
import_size = os.path.getsize(import_output_path) / (1024 * 1024)  # 转换为MB
export_size = os.path.getsize(export_output_path) / (1024 * 1024)  # 转换为MB
print(f"\n文件大小：")
print(f"Import数据：{import_size:.2f} MB")
print(f"Export数据：{export_size:.2f} MB")

# 显示最终的列名
print("\n最终的Import数据列名：")
print(import_data.columns.tolist())
print("\n最终的Export数据列名：")
print(export_data.columns.tolist())

=== 数据保存完成 ===
Import数据已保存到：../data/interim/combined_import_data.csv
Export数据已保存到：../data/interim/combined_export_data.csv

文件大小：
Import数据：0.20 MB
Export数据：0.13 MB

最终的Import数据列名：
['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'US dollar', 'source_file']

最终的Export数据列名：
['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'US dollar', 'source_file']


## 2. 数据加载与预处理

In [6]:
# 2. 数据加载
def load_trade_data(file_pattern):
    """
    加载并合并贸易数据文件
    
    参数:
        file_pattern: 文件匹配模式（'*Import.csv' 或 '*Export.csv'）
    返回:
        合并后的数据框
    """
    files = glob.glob(os.path.join(RAW_DATA_PATH, file_pattern))
    dfs = []
    
    for file in files:
        try:
            # 尝试使用不同的编码读取
            for encoding in ['utf-8', 'gbk', 'gb2312', 'gb18030']:
                try:
                    df = pd.read_csv(file, encoding=encoding)
                    break
                except UnicodeDecodeError:
                    continue
            else:
                print(f"警告: 无法读取文件 {os.path.basename(file)}")
                continue
            
            # 检查必要的列是否存在
            if 'Date of data' not in df.columns:
                print(f"警告: {os.path.basename(file)} 中没有 'Date of data' 列")
                continue
            
            # 添加来源文件信息
            df['source_file'] = os.path.basename(file)
            
            dfs.append(df)
            print(f"成功加载文件: {os.path.basename(file)}")
            
        except Exception as e:
            print(f"处理文件 {os.path.basename(file)} 时出错: {str(e)}")
            continue
    
    if not dfs:
        raise ValueError(f"没有找到匹配的文件: {file_pattern}")
    
    # 合并所有数据
    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df

# 加载进口和出口数据
print("=== 开始加载数据 ===")
import_data = load_trade_data("*Import.csv")
export_data = load_trade_data("*Export.csv")

# 显示数据基本信息
print("\n=== 数据加载完成 ===")
print(f"进口数据形状: {import_data.shape}")
print(f"出口数据形状: {export_data.shape}")
print("\n进口数据列：", import_data.columns.tolist())
print("\n出口数据列：", export_data.columns.tolist())
print("\n=== 数据预览（前5行）===")
print("\n进口数据前5行：")
print(import_data.head())
print("\n出口数据前5行：")
print(export_data.head())

=== 开始加载数据 ===
成功加载文件: 2024_China Import.csv
成功加载文件: 202508_China Import.csv
成功加载文件: 202501&06_China Import.csv
成功加载文件: 202507_China Import.csv
成功加载文件: 2020_China Import.csv
成功加载文件: 202509_China Import.csv
成功加载文件: 2021_China Import.csv
成功加载文件: 2022_China Import.csv
成功加载文件: 2023_China Import.csv
成功加载文件: 2021_China Export.csv
成功加载文件: 202509_China Export.csv
成功加载文件: 2020_China Export.csv
成功加载文件: 202501&06_China Export.csv
成功加载文件: 202507_China Export.csv
成功加载文件: 2024_China Export.csv
成功加载文件: 202508_China Export.csv
成功加载文件: 2023_China Export.csv
成功加载文件: 2022_China Export.csv

=== 数据加载完成 ===
进口数据形状: (1776, 12)
出口数据形状: (1164, 12)

进口数据列： ['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'US dollar', 'Unnamed: 10', 'source_file']

出口数据列： ['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary

## 1. 环境准备

In [1]:
# 1. 环境准备
# 导入所需库
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path
from calendar import monthrange
from datetime import datetime

# 设置数据路径
RAW_DATA_PATH = "../data/raw"
PROCESSED_DATA_PATH = "../data/processed"

# 确保processed文件夹存在
Path(PROCESSED_DATA_PATH).mkdir(parents=True, exist_ok=True)

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 中国进出口数据分析与聚合

本 notebook 用于处理和聚合中国进出口贸易数据。主要流程包括：

1. 环境准备
   - 导入必要的库
   - 设置文件路径
   
2. 数据加载与预处理
   - 读取所有进口数据文件
   - 读取所有出口数据文件
   - 数据清洗和标准化
   - 时间信息提取
   
3. 数据聚合与分析
   - 数据标准化
   - 处理缺失值
   - 格式统一化
   
4. 结果保存
   - 保存处理后的数据
   - 数据统计汇总

In [ ]:
# 4. 保存处理后的数据
def save_data(df, filename):
    """
    保存数据到CSV文件
    
    参数:
        df: 要保存的数据框
        filename: 输出文件名
    """
    output_path = os.path.join(PROCESSED_DATA_PATH, filename)
    df.to_csv(output_path, index=False)
    print(f"数据已保存至: {output_path}")
    print(f"文件大小: {os.path.getsize(output_path) / 1024:.2f} KB")
    print(f"行数: {len(df)}")
    print(f"列数: {len(df.columns)}")

# 保存处理后的数据
print("=== 保存进口数据 ===")
save_data(processed_import_data, 'china_import_data.csv')

print("\n=== 保存出口数据 ===")
save_data(processed_export_data, 'china_export_data.csv')

print("\n数据处理完成！")